# Import Required libraries

In [1]:
# Python basic packages
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn packages
from sklearn.model_selection import train_test_split


# tensorflow packages
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import (Dense, 
                                    Input, 
                                    LSTM,
                                    Dropout)
from tensorflow.keras.optimizers import (Adam, 
                                         AdamW)
from tensorflow.keras.callbacks import (EarlyStopping, 
                                        ModelCheckpoint)
from tensorflow.keras.losses import (SparseCategoricalCrossentropy,
                                     CategoricalCrossentropy)                                         

/Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/tensorflow_env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# Load Train, Validation and Test Dataset

In [2]:
X_train = pd.read_csv("./processed_data/RMS_X_train_features.csv").drop(columns = ["SubjectID", "Motor_label"])
X_test = pd.read_csv("./processed_data/RMS_X_test_features.csv").drop(columns = ["SubjectID", "Motor_label"])

In [3]:
X_test

,Label,RMS_Feature_1,RMS_Feature_2,RMS_Feature_3,RMS_Feature_4,RMS_Feature_5,RMS_Feature_6,RMS_Feature_7,RMS_Feature_8,RMS_Feature_9,RMS_Feature_10,RMS_Feature_11,RMS_Feature_12,RMS_Feature_13,RMS_Feature_14,RMS_Feature_15
0,1,0.538470,0.549012,0.585014,0.646651,0.561444,0.625809,0.575417,0.534146,0.641276,0.419086,0.610804,0.704755,0.522936,0.632867,0.696071
1,2,0.523831,0.512782,0.496030,0.483269,0.511846,0.531214,0.545420,0.517940,0.503801,0.532118,0.563648,0.534215,0.541327,0.527049,0.530629
2,3,0.522992,0.514977,0.545879,0.533157,0.523995,0.503702,0.514512,0.478064,0.571765,0.513028,0.530894,0.521043,0.516542,0.518184,0.530536
3,4,0.494842,0.540550,0.532703,0.549623,0.548304,0.521287,0.529149,0.527342,0.518213,0.529674,0.514686,0.530861,0.533082,0.518601,0.543139
4,5,0.517465,0.521787,0.531081,0.538597,0.533957,0.519866,0.536361,0.535086,0.566026,0.519034,0.501186,0.498468,0.510947,0.561301,0.527791
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,3,0.502594,0.498184,0.521599,0.525810,0.494094,0.519277,0.517928,0.533611,0.553843,0.466682,0.501285,0.497462,0.535093,0.539392,0.538149
80,4,0.530964,0.530241,0.542248,0.542795,0.554361,0.534638,0.531712,0.543553,0.479973,0.527828,0.503037,0.524312,0.507423,0.526953,0.522959
81,5,0.509227,0.510610,0.517647,0.524551,0.520735,0.524067,0.524426,0.553883,0.504940,0.524833,0.550713,0.544592,0.559718,0.519850,0.507670
82,6,0.486879,0.568341,0.512567,0.536030,0.540502,0.545280,0.502414,0.498549,0.486667,0.541269,0.532308,0.513379,0.503323,0.489462,0.474606


In [4]:
X_train, X_val, y_train, y_val = train_test_split(X_train.drop("Label", axis = 1), X_train["Label"], train_size=0.8, random_state=42)
print(f"Shape of X_train : {X_train.shape}")
print(f"Shape of X_train : {X_val.shape}")
print(f"Shape of X_train : {y_train.shape}")
print(f"Shape of X_train : {y_val.shape}")

Shape of X_train : (268, 15)
Shape of X_train : (68, 15)
Shape of X_train : (268,)
Shape of X_train : (68,)


# Create tf.Dataset

In [5]:
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, to_categorical(y_train))).batch(64)
validation_dataset = tf.data.Dataset.from_tensor_slices((X_val, to_categorical(y_val))).batch(64)

2025-08-03 00:22:51.534167: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
2025-08-03 00:22:51.534194: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
2025-08-03 00:22:51.534200: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.00 GB
I0000 00:00:1754198571.534210 1235819 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1754198571.534230 1235819 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [6]:
test_dataset = tf.data.Dataset.from_tensor_slices((X_test.drop("Label", axis = 1))).batch(64)

# Neural Network Model

#### Step 1: Baseline Model Architecture

In [7]:
class SimpleDLModel(Model):
    def __init__(self, num_classes):
        super(SimpleDLModel, self).__init__()
        self.dense1 = Dense(128, activation='leaky_relu')
        self.dropout1 = Dropout(0.2)
        self.dense2 = Dense(64, activation='leaky_relu')
        self.dropout2 = Dropout(0.2)
        self.output_layer = Dense(num_classes, activation='softmax')

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dropout1(x)
        x = self.dense2(x)
        x = self.dropout2(x)
        return self.output_layer(x)
    
SimpleDLModel = SimpleDLModel(num_classes=8)   

#### Step 2: Compile the Model

In [8]:
SimpleDLModel.compile(optimizer=AdamW(learning_rate=0.001),
                      loss=CategoricalCrossentropy(from_logits=False),
                      metrics=['f1_score', 'accuracy'])

#### Step 3: Train the Model

In [9]:
SimpleDLModel.fit(train_dataset,
                 validation_data=validation_dataset,
                 epochs=100,
                 )

Epoch 1/100


2025-08-03 00:22:51.860235: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 85ms/step - accuracy: 0.1582 - f1_score: 0.0544 - loss: 2.0404 - val_accuracy: 0.1765 - val_f1_score: 0.0385 - val_loss: 2.0016
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1637 - f1_score: 0.0579 - loss: 1.9940 - val_accuracy: 0.0735 - val_f1_score: 0.0187 - val_loss: 2.0007
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.1495 - f1_score: 0.0569 - loss: 1.9826 - val_accuracy: 0.1176 - val_f1_score: 0.0278 - val_loss: 1.9987
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1547 - f1_score: 0.0411 - loss: 1.9792 - val_accuracy: 0.1324 - val_f1_score: 0.0444 - val_loss: 1.9915
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.1608 - f1_score: 0.0504 - loss: 1.9778 - val_accuracy: 0.1912 - val_f1_score: 0.0856 - val_loss: 1.9820
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1484 - f1_score: 0.0547 - loss: 1.9749 - val_accuracy: 0.1765 - val_f1_score: 0.0698 - val_loss: 1.9731
Epoc

# Prediction on test dataset

In [14]:
test_predictions= np.argmax(SimpleDLModel.predict(test_dataset), axis=-1)
test_predictions = pd.DataFrame(test_predictions, columns=["Predicted_Label"])
test_predictions["Label"] = X_test["Label"].values

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


#### Metrics and Evaluation

In [18]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(test_predictions["Label"], test_predictions["Predicted_Label"])
cm

array([[1, 3, 0, 0, 5, 0, 3],
       [0, 3, 0, 0, 6, 0, 3],
       [0, 5, 0, 0, 4, 0, 3],
       [0, 2, 0, 0, 6, 0, 4],
       [0, 5, 0, 0, 7, 0, 0],
       [0, 5, 0, 0, 5, 0, 2],
       [0, 3, 0, 0, 6, 0, 3]])